# 04 — Baseline modeling (run only after leakage review)

**Gate:** do not run this notebook until Notebook 02's leakage decisions and Notebook 03's feature provenance register are complete. This notebook provides a baseline, not a final research claim.

## 1. Load data and configure approved features

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset(1).csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

In [ ]:
TARGET = "anomaly"

# Replace this list with ONLY the features approved in the provenance register.
# The default below excludes known/suspected label proxies and is deliberately conservative.
EXCLUDE = {
    TARGET, "risk_score", "transaction_type", "purchase_pattern", "age_group"
}
FEATURES = [c for c in df.columns if c not in EXCLUDE]

print("Features used:", FEATURES)
print("Target distribution:")
display(df[TARGET].value_counts())

## 2. Chronological split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier

work = df[FEATURES + [TARGET]].copy()
if "timestamp" in work.columns:
    work["_sort_time"] = pd.to_datetime(work["timestamp"], errors="coerce", utc=True)
    work = work.sort_values("_sort_time").drop(columns="_sort_time")
    cut = int(len(work) * 0.8)
    train, test = work.iloc[:cut], work.iloc[cut:]
else:
    train, test = train_test_split(work, test_size=0.2, random_state=42, stratify=work[TARGET])

X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train[TARGET].astype(str), test[TARGET].astype(str)

## 3. Leakage-safe preprocessing and baseline model

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in FEATURES if c not in num_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=42, n_jobs=-1
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix (label order):", model.classes_)
display(pd.DataFrame(
    confusion_matrix(y_test, pred, labels=model.classes_),
    index=model.classes_, columns=model.classes_
))

## 4. Save evaluation outputs

In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
pd.DataFrame(classification_report(y_test, pred, output_dict=True, zero_division=0)).T.to_csv(
    "outputs/04_classification_report.csv"
)
pd.DataFrame(
    confusion_matrix(y_test, pred, labels=model.classes_),
    index=model.classes_, columns=model.classes_
).to_csv("outputs/04_confusion_matrix.csv")
print("Saved outputs/04_classification_report.csv and outputs/04_confusion_matrix.csv")

## Interpretation requirements

- Report the split method and exact dates.
- Report class distribution in train and test.
- Compare against a majority-class baseline.
- Include per-class precision, recall, F1, confusion matrix, and PR-AUC where appropriate.
- Repeat with a justified alternative feature set only after provenance review.
- Do not claim deployment readiness from a single random-forest run.